# The sizing loop in MATLAB

This page runs the sizing iteration of {numref}`fig-sizing-loop` on the previous page, cell by cell. The code is MATLAB. The outputs shown were produced by MATLAB R2026a when the page was built.

:::{tip} Run and edit the cells in your browser
Click the launch button at the top of the page and wait for the kernel to connect. Then run the cells in order with their run buttons. Change an input and run again to see the effect. The kernel is GNU Octave on mybinder.org, a free public service, and it runs this MATLAB code unchanged. The first start can take a few minutes while the environment builds. To run the code on your own computer in MATLAB, download `sizing_loop.m` from the previous page.
:::

:::{warning} Example values
The inputs below describe a small jet transport with invented values, so that the example is not the Navy Jet Trainer. Your sizing code takes its mission from your RFP and its sizing variables from your constraint analysis.
:::

## Inputs

Fixed weights and the design mission.

In [1]:
% Fixed weights, lb (invented values for a small jet transport)
W_crew    = 600;      % three crew with equipment
W_payload = 8000;     % 40 passengers with bags, 200 lb each

% Design mission (example values)
R_cruise  = 1500 * 6076;    % cruise range, ft (1,500 nmi)
V_cruise  = 450 * 1.688;    % cruise true airspeed, ft/s (450 kt)
C_cruise  = 0.60 / 3600;    % thrust specific fuel consumption in cruise, 1/s
LD_cruise = 15;             % lift-to-drag ratio in cruise
E_loiter  = 20 * 60;        % loiter endurance, s (20 min)
C_loiter  = 0.60 / 3600;    % thrust specific fuel consumption in loiter, 1/s
LD_loiter = 16;             % lift-to-drag ratio in loiter
k         = 0.06;           % trapped fuel 1 percent plus 5 percent reserves

## Empty weight

The bomber and transport trend line of Nicolai and Carichner (2010), Table 5.1. It is a function of the takeoff weight, which is why the loop is needed.

In [2]:
We_of_W0 = @(W0) 0.911 * W0.^0.947;
fprintf('We/W0 at 20,000 lb: %.3f   at 40,000 lb: %.3f\n', We_of_W0(20000)/20000, We_of_W0(40000)/40000)

We/W0 at 20,000 lb: 0.539   at 40,000 lb: 0.520


## Fuel weight fraction

The product of the segment fractions: Raymer's historical fractions for takeoff, climb, and landing, and the Breguet range and endurance equations for cruise and loiter.

In [3]:
f_takeoff = 0.970;                                               % Raymer Table 3.2
f_climb   = 0.985;                                               % Raymer Table 3.2
f_cruise  = exp(-R_cruise * C_cruise / (V_cruise * LD_cruise));  % Breguet range
f_loiter  = exp(-E_loiter * C_loiter / LD_loiter);               % Breguet endurance
f_landing = 0.995;                                               % Raymer Table 3.2
Wfuel_frac = (1 + k) * (1 - f_takeoff * f_climb * f_cruise * f_loiter * f_landing);
fprintf('Cruise fraction %.4f, loiter fraction %.4f, fuel weight fraction Wfuel/W0 = %.3f\n', ...
    f_cruise, f_loiter, Wfuel_frac)

Cruise fraction 0.8752, loiter fraction 0.9876, fuel weight fraction Wfuel/W0 = 0.189


## The loop

Guess $W_0$, compute the empty weight and the fuel weight from the guess, and solve the gross-weight sum for a new $W_0$. The sum $W_0 = W_{crew} + W_{payload} + W_e + W_{fuel}$ is rearranged as $W_0 = (W_{crew} + W_{payload}) / (1 - W_e/W_0 - W_{fuel}/W_0)$, so each iteration moves most of the way to the answer.

In [4]:
W0_guess  = 20000;     % lb, initial guess
tolerance = 1;         % lb
fprintf('%5s %12s %12s %12s %12s\n', 'iter', 'W0 guess', 'We', 'Wfuel', 'W0 calc');
for iteration = 1:100
    We    = We_of_W0(W0_guess);
    Wfuel = Wfuel_frac * W0_guess;
    W0_calc = (W_crew + W_payload) / (1 - We / W0_guess - Wfuel / W0_guess);
    fprintf('%5d %12.0f %12.0f %12.0f %12.0f\n', iteration, W0_guess, We, Wfuel, W0_calc);
    if abs(W0_calc - W0_guess) < tolerance
        break
    end
    W0_guess = W0_calc;    % (W0)guess = (W0)calc, and back to the top
end
fprintf('\nConverged after %d iterations: W0 = %.0f lb (We/W0 = %.3f, Wfuel/W0 = %.3f)\n', ...
    iteration, W0_calc, We_of_W0(W0_calc) / W0_calc, Wfuel_frac)

 iter     W0 guess           We        Wfuel      W0 calc
    1        20000        10779         3780        31616
    2        31616        16631         5976        30182
    3        30182        15916         5705        30320
    4        30320        15985         5731        30306
    5        30306        15978         5728        30308
    6        30308        15979         5729        30307

Converged after 6 iterations: W0 = 30307 lb (We/W0 = 0.527, Wfuel/W0 = 0.189)


The loop converges in a few iterations because the empty-weight fraction changes slowly with $W_0$. With the Raymer regression in place of the trend line, the fraction also depends on the sizing variables, and the same loop still works.